# Calibrated MaxEnt module experiments — Colab

This notebook trains class-specialized sparse MaxEnt modules
`M(class, seed)` and compares them across three complementary levels:

1. **Structural/support level** — sparse masks and weights, including
   permutation-aligned IoU and a matching-free spectral mask distance.
2. **Representational level** — CKA, Orthogonal Procrustes, activation
   correlations, and participation ratio on the same fixed evaluation set.
3. **Functional level** — output MSE/KL, rewarded accuracy, non-rewarded
   entropy, and module composition by weight summation.

The motivating question is whether modules trained for the same rewarded class
are more similar than modules trained for different classes, once neuron
permutation confounds are controlled.

**Before running**
1. `Runtime -> Change runtime type -> GPU` (T4 is enough for this MLP).
2. Run cells top to bottom. In Colab, the storage cell will ask for Google Drive
   authorization. In Kaggle, artifacts are written under `/kaggle/working/tesi`.
3. Keep the first run as `SMOKE = True`; only switch to the full run after all
   smoke-test calibration controls print `[controls] PASS`.

**Persistence and resume behavior**

All artifacts are written under a persistent output directory: `MyDrive/tesi/`
on Colab, `/kaggle/working/tesi/` on Kaggle, or `raw/experiments/` when run
locally. If the runtime disconnects, re-run the notebook with the same
configuration: completed modules are skipped automatically unless
`args.overwrite = True`.

**Full 10x10 run**

For the complete run with 10 classes and 10 seeds, set:

```python
SMOKE = False
FULL_10X10 = True
```

The notebook also keeps a smaller 5 classes x 3 seeds option for quick checks.
Interpolation barriers are disabled by default in real runs because they are
optional and expensive on 100 modules; enable them only for a targeted follow-up.


## Environment check

Prints package versions and confirms whether the runtime sees a GPU.

In [ ]:
# This cell checks the runtime, package versions, and selected accelerator.
import sys
import torch, scipy, numpy, matplotlib

try:
    import torchvision
    torchvision_version = torchvision.__version__
except Exception as exc:
    torchvision_version = f"unavailable ({exc})"

print("python ", sys.version.split()[0])
print("torch  ", torch.__version__, "| torchvision", torchvision_version)
print("scipy  ", scipy.__version__, "| numpy", numpy.__version__)
print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## Output directory

Detects Colab, Kaggle, or local execution and stores all artifacts in the matching output folder.

In [ ]:
# This cell detects Colab/Kaggle/local execution and chooses the output directory.
from pathlib import Path
import os

try:
    import google.colab  # noqa: F401
    # Kaggle may have the google.colab package installed, but it does not support
    # drive.mount. A real Colab runtime has this marker file.
    IN_COLAB = Path("/var/colab/hostname").exists()
except ImportError:
    IN_COLAB = False

IN_KAGGLE = Path("/kaggle/working").exists() and not IN_COLAB

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/tesi")
elif IN_KAGGLE:
    BASE_DIR = Path("/kaggle/working/tesi")
else:
    BASE_DIR = Path("raw/experiments")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  IN_KAGGLE={IN_KAGGLE}  artifacts -> {BASE_DIR.resolve()}")


## Pipeline code

The cells below are generated verbatim from
`scripts/run_calibrated_module_experiments.py` (smoke-tested: all 8
calibration controls PASS). **Do not edit these pipeline cells in Colab.**
Change only the *Configuration* cell below. If the Python script changes,
regenerate the notebook with `python3 scripts/make_colab_notebook.py`.


### Imports & constants

Imports libraries and defines shared constants used by the full pipeline.

In [ ]:
#!/usr/bin/env python3
"""Train MaxEnt sparse modules and compare them with permutation-calibrated metrics.

This script implements the experimental setting described in
``drafts/setting_esatto_test_sperimentali_moduli.md``. It

  1. trains class-specialized sparse modules  M(class, seed)  with a
     Maximum-Entropy (MaxEnt) loss and Iterative Magnitude Pruning (IMP);
  2. saves every reproducibility artifact (checkpoint, mask, config, metrics,
     activations, labels, eval indices);
  3. forms all module pairs, labels each pair with a category
     (identical / permuted_control / random_baseline / same_class_diff_seed /
     diff_class_same_seed / diff_class_diff_seed);
  4. computes structural, representational and functional metrics, using
     permutation-aligned variants that are *not* confounded by hidden-neuron
     relabeling (Git Re-Basin style weight-matching and activation-matching),
     plus: mask spectral distance (matching-free permutation-invariant control),
     the analytic IoU null at matched density, CKA/Procrustes decomposed by
     rewarded vs non-rewarded inputs, pairwise composition (theta_a + theta_b,
     Caso et al. 3.3.1) with interference, and optional interpolation barriers
     raw vs aligned (--barrier; Entezari et al. 2022 / Ainsworth et al. 2023);
  5. aggregates by pair category / layer / class pair, runs significance tests
     (Mann-Whitney + module-label permutation test), writes JSON + CSV,
     produces the required figures, and writes a concise and a full report.

The metric implementations follow ``scripts/metric_sanity_check_model.py``
(the controlled permuted-MLP sanity check) so that the calibration validated
there is reused here on real trained modules.

Run ``python scripts/run_calibrated_module_experiments.py --help`` for options.
A tiny end-to-end run is available with ``--smoke-test``.
"""

from __future__ import annotations

import argparse
import csv
import gzip
import json
import os
import random
import struct
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-calibrated")

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Subset, TensorDataset
except Exception as exc:  # pragma: no cover - torch is required
    print(f"ERROR: PyTorch is required to run this script: {exc}", file=sys.stderr)
    raise

from scipy.optimize import linear_sum_assignment

EPS = 1e-12
LAYERS = ("fc1", "fc2", "fc3")
HIDDEN_LAYERS = ("fc1", "fc2")  # layers whose output units can be permuted
PAIR_CATEGORIES = (
    "identical",
    "permuted_control",
    "random_baseline",
    "same_class_diff_seed",
    "diff_class_same_seed",
    "diff_class_diff_seed",
)

### Reproducibility

Sets random seeds and timestamp helpers so module training can be reproduced.

In [ ]:
def seed_everything(seed: int) -> None:
    """Seed all relevant RNGs for deterministic-as-possible behaviour on CPU."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

### Data

Loads MNIST, fixes the shared evaluation subset, and materializes tensors for training/evaluation.

In [ ]:
def load_mnist(data_root: Path, train: bool):
    """Return MNIST with the same preprocessing in every environment.

    Colab normally has ``torchvision`` available, so the standard dataset loader
    is the preferred path. The local thesis workspace may only have the raw IDX
    files; in that case the fallback below reads those files directly and applies
    the same ToTensor + MNIST normalization convention.
    """
    try:
        from torchvision import datasets, transforms

        transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        return datasets.MNIST(str(data_root), train=train, download=True, transform=transform)
    except Exception:
        return load_mnist_idx(data_root, train=train)


def _open_idx(path: Path):
    if path.exists():
        return path.open("rb")
    gz_path = path.with_suffix(path.suffix + ".gz")
    if gz_path.exists():
        return gzip.open(gz_path, "rb")
    raise FileNotFoundError(f"missing MNIST IDX file: {path}(.gz)")


def _read_idx_images(path: Path) -> np.ndarray:
    with _open_idx(path) as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"unexpected MNIST image magic in {path}: {magic}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols)


def _read_idx_labels(path: Path) -> np.ndarray:
    with _open_idx(path) as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"unexpected MNIST label magic in {path}: {magic}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n)


def load_mnist_idx(data_root: Path, train: bool) -> TensorDataset:
    """Load cached MNIST IDX files directly when torchvision is unavailable.

    This keeps real runs on MNIST even when torchvision is unavailable.
    """
    raw_dir = data_root / "MNIST" / "raw"
    split = "train" if train else "t10k"
    images = _read_idx_images(raw_dir / f"{split}-images-idx3-ubyte")
    labels = _read_idx_labels(raw_dir / f"{split}-labels-idx1-ubyte")
    x = torch.from_numpy(images.astype(np.float32)).unsqueeze(1) / 255.0
    x = (x - 0.1307) / 0.3081
    y = torch.from_numpy(labels.astype(np.int64))
    return TensorDataset(x, y)


def get_datasets(args) -> tuple[Any, Any, str]:
    """Return MNIST train/test datasets or fail loudly.

    The thesis-facing experiment is defined on MNIST. If the dataset cannot be
    loaded, the correct behaviour is to stop and fix the cache/download rather
    than accidentally producing results on another distribution.
    """
    try:
        train = load_mnist(args.data_root, train=True)
        test = load_mnist(args.data_root, train=False)
        return train, test, "MNIST"
    except Exception as exc:  # offline / download failure
        raise RuntimeError(
            "Could not load MNIST. Fix the MNIST download/cache and rerun; "
            "this experiment intentionally does not substitute another dataset."
        ) from exc


def stack_dataset(dataset) -> tuple[torch.Tensor, torch.Tensor]:
    """Materialise a dataset to flat tensors X (N,784) and y (N,)."""
    loader = DataLoader(dataset, batch_size=1024, shuffle=False)
    xs, ys = [], []
    for x, y in loader:
        xs.append(x.view(x.size(0), -1))
        ys.append(y)
    return torch.cat(xs), torch.cat(ys)


def build_eval_subset(
    test_x: torch.Tensor, test_y: torch.Tensor, n_examples: int, seed: int
) -> tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Pick a FIXED subset of test examples shared by every module.

    The same rows (and order) are used for all modules so that activation
    matrices are directly comparable.
    """
    n_total = test_x.size(0)
    n = min(n_examples, n_total)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(n_total, size=n, replace=False))
    return test_x[idx], test_y[idx], idx

### Model

Defines the DeepMLP architecture and activation collection points used by the metrics.

In [ ]:
class DeepMLP(nn.Module):
    """784 -> 512 -> 256 -> 10 MLP (ReLU on hidden layers, logits at output)."""

    def __init__(self, hidden1: int = 512, hidden2: int = 256, num_outputs: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_outputs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    @torch.no_grad()
    def collect_activations(self, x: torch.Tensor) -> dict[str, np.ndarray]:
        """Return the Linear-layer outputs (pre-ReLU for hidden, logits for fc3).

        These are the representations compared by CKA/Procrustes, matching the
        convention used in the prior Exp1b analysis (hooks on the Linear modules).
        """
        x = x.view(x.size(0), -1)
        z1 = self.fc1(x)
        z2 = self.fc2(F.relu(z1))
        z3 = self.fc3(F.relu(z2))
        return {
            "fc1": z1.cpu().numpy(),
            "fc2": z2.cpu().numpy(),
            "fc3": z3.cpu().numpy(),
        }


def layer_modules(model: DeepMLP) -> dict[str, nn.Linear]:
    return {"fc1": model.fc1, "fc2": model.fc2, "fc3": model.fc3}

### MaxEnt loss

Implements the class-specialized MaxEnt objective: confident on the rewarded class, high entropy elsewhere.

In [ ]:
def maxent_loss(
    logits: torch.Tensor, targets: torch.Tensor, rewarded_class: int, lam: float
) -> torch.Tensor:
    """MaxEnt loss of Caso et al. (2026).

    * rewarded examples (y == rewarded_class): standard cross-entropy toward the
      rewarded class -> confident prediction.
    * non-rewarded examples (y != rewarded_class): push the softmax toward the
      uniform distribution -> maximum entropy (~ ln(num_classes)). Implemented as
      soft cross-entropy against a uniform target, which equals KL(uniform||p) up
      to a constant and drives all logits to be equal.

    The two terms are averaged within their own group and combined with weight
    ``lam`` on the MaxEnt (non-rewarded) term.
    """
    log_p = F.log_softmax(logits, dim=1)
    is_rew = targets == rewarded_class

    loss = logits.new_zeros(())
    if is_rew.any():
        loss = loss + F.nll_loss(log_p[is_rew], targets[is_rew])
    if (~is_rew).any():
        # -mean_k log p_k  averaged over non-rewarded examples
        loss = loss + lam * (-log_p[~is_rew].mean(dim=1).mean())
    return loss

### Iterative Magnitude Pruning

Trains sparse modules with reset-to-initialization IMP and saves the final sparse support.

In [ ]:
def ones_like_masks(model: DeepMLP) -> dict[str, torch.Tensor]:
    return {name: torch.ones_like(m.weight) for name, m in layer_modules(model).items()}


def apply_masks(model: DeepMLP, masks: dict[str, torch.Tensor]) -> None:
    with torch.no_grad():
        for name, m in layer_modules(model).items():
            m.weight.mul_(masks[name].to(m.weight.device))


def magnitude_prune(
    model: DeepMLP, masks: dict[str, torch.Tensor], rate: float
) -> dict[str, torch.Tensor]:
    """Global unstructured magnitude pruning: remove ``rate`` of remaining weights."""
    mods = layer_modules(model)
    survivors = []
    for name, m in mods.items():
        active = masks[name].bool()
        survivors.append(m.weight.detach().abs()[active].flatten())
    all_active = torch.cat(survivors)
    if all_active.numel() == 0:
        return masks
    k = int(rate * all_active.numel())
    if k <= 0:
        return masks
    threshold = torch.kthvalue(all_active, k).values.item()
    new_masks = {}
    for name, m in mods.items():
        keep = (m.weight.detach().abs() > threshold) & masks[name].bool()
        new_masks[name] = keep.float()
    return new_masks


def grad_freeze_pruned(model: DeepMLP, masks: dict[str, torch.Tensor]) -> None:
    """Zero out gradients of pruned weights so they stay at 0 during training."""
    for name, m in layer_modules(model).items():
        if m.weight.grad is not None:
            m.weight.grad.mul_(masks[name].to(m.weight.device))


def train_loop(
    model: DeepMLP,
    masks: dict[str, torch.Tensor],
    loader: DataLoader,
    rewarded_class: int,
    lam: float,
    epochs: int,
    lr: float,
    device: str,
) -> list[float]:
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history: list[float] = []
    for _ in range(epochs):
        running, n_batches = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = maxent_loss(logits, y, rewarded_class, lam)
            loss.backward()
            grad_freeze_pruned(model, masks)
            optimizer.step()
            apply_masks(model, masks)  # keep pruned weights exactly 0
            running += float(loss.item())
            n_batches += 1
        history.append(running / max(n_batches, 1))
    return history


def imp_train_module(
    rewarded_class: int,
    seed: int,
    train_loader_factory,
    args,
    device: str,
) -> tuple[DeepMLP, dict[str, torch.Tensor], dict[str, Any]]:
    """Train one module with reset-to-init IMP (classic Lottery-Ticket schedule).

    For each of ``prune_rounds`` rounds: reset weights to the initial values,
    apply the current mask, train, then prune ``prune_rate`` of the surviving
    weights. A final fit is then performed at the target sparsity so that the
    saved checkpoint is a trained sparse network whose mask has density
    (1 - prune_rate) ** prune_rounds.
    """
    seed_everything(seed)
    model = DeepMLP().to(device)
    theta_0 = {k: v.detach().clone() for k, v in model.state_dict().items()}
    masks = ones_like_masks(model)

    # IMP schedule used here: train at the current mask, prune the lowest
    # magnitude surviving weights, then reset surviving weights to the original
    # initialization. This separates "which sparse support was found" from the
    # temporary weights used to decide the next pruning step.
    loss_curves: list[list[float]] = []
    for _round in range(args.prune_rounds):
        model.load_state_dict(theta_0)
        apply_masks(model, masks)
        loader = train_loader_factory(seed + _round)
        history = train_loop(
            model, masks, loader, rewarded_class, args.lambda_maxent,
            args.epochs_per_round, args.lr, device,
        )
        loss_curves.append(history)
        masks = magnitude_prune(model, masks, args.prune_rate)

    # Final fit at target sparsity. This is the checkpoint used by all pairwise
    # structural, representational, and functional comparisons.
    model.load_state_dict(theta_0)
    apply_masks(model, masks)
    final_loader = train_loader_factory(seed + args.prune_rounds)
    final_history = train_loop(
        model, masks, final_loader, rewarded_class, args.lambda_maxent,
        args.epochs_per_round, args.lr, device,
    )
    loss_curves.append(final_history)

    info = {"loss_curves": loss_curves}
    return model, masks, info

### Functional evaluation

Computes per-module rewarded accuracy, non-rewarded entropy, and sparsity summaries.

In [ ]:
@torch.no_grad()
def evaluate_module(
    model: DeepMLP, eval_x: torch.Tensor, eval_y: torch.Tensor,
    rewarded_class: int, device: str,
) -> dict[str, float]:
    model.eval()
    logits = model(eval_x.to(device))
    probs = F.softmax(logits, dim=1)
    log_probs = F.log_softmax(logits, dim=1)
    entropy = -(probs * log_probs).sum(dim=1)  # per-example entropy

    rew_mask = eval_y == rewarded_class
    nonrew_mask = ~rew_mask
    preds = logits.argmax(dim=1)

    rewarded_acc = (
        float((preds[rew_mask] == rewarded_class).float().mean()) if rew_mask.any() else float("nan")
    )
    nonrew_entropy = float(entropy[nonrew_mask].mean()) if nonrew_mask.any() else float("nan")
    return {
        "rewarded_accuracy": rewarded_acc,
        "nonrewarded_entropy": nonrew_entropy,
        "mean_entropy_all": float(entropy.mean()),
        "n_rewarded_eval": int(rew_mask.sum()),
    }


def layer_sparsity(masks: dict[str, torch.Tensor]) -> dict[str, float]:
    out: dict[str, float] = {}
    tot_active, tot = 0, 0
    for name, m in masks.items():
        active = int(m.sum().item())
        total = int(m.numel())
        out[name] = 1.0 - active / total
        tot_active += active
        tot += total
    out["global"] = 1.0 - tot_active / tot
    return out

### Module container & persistence

Defines the module data structure and reads/writes checkpoints, masks, configs, metrics, and activations.

In [ ]:
@dataclass
class Module:
    cls: int
    seed: int
    weights: dict[str, np.ndarray]  # fc1,fc2,fc3 -> (out, in)
    biases: dict[str, np.ndarray]  # fc1,fc2,fc3 -> (out,)
    masks: dict[str, np.ndarray]  # fc1,fc2,fc3 -> bool (out, in)
    acts: dict[str, np.ndarray]  # fc1,fc2,fc3 -> (N, p)
    functional: dict[str, float]
    sparsity: dict[str, float]

    @property
    def key(self) -> str:
        return f"class{self.cls}_seed{self.seed}"

    @property
    def logits(self) -> np.ndarray:
        return self.acts["fc3"]


def module_paths(out_dir: Path, cls: int, seed: int) -> dict[str, Path]:
    tag = f"class{cls}_seed{seed}"
    return {
        "checkpoint": out_dir / "checkpoints" / f"module_{tag}.pt",
        "mask": out_dir / "masks" / f"mask_{tag}.pt",
        "config": out_dir / "configs" / f"config_{tag}.json",
        "metrics": out_dir / "metrics" / f"train_eval_{tag}.json",
        "act_fc1": out_dir / "activations" / f"{tag}_fc1.npy",
        "act_fc2": out_dir / "activations" / f"{tag}_fc2.npy",
        "act_fc3": out_dir / "activations" / f"{tag}_fc3.npy",
    }


def module_is_complete(out_dir: Path, cls: int, seed: int) -> bool:
    return all(p.exists() for p in module_paths(out_dir, cls, seed).values())


def save_module(
    out_dir: Path, model: DeepMLP, masks: dict[str, torch.Tensor],
    cls: int, seed: int, functional: dict[str, float], sparsity: dict[str, float],
    acts: dict[str, np.ndarray], info: dict[str, Any], args, data_source: str,
) -> None:
    paths = module_paths(out_dir, cls, seed)
    for p in paths.values():
        p.parent.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), paths["checkpoint"])
    torch.save({k: v.bool() for k, v in masks.items()}, paths["mask"])
    np.save(paths["act_fc1"], acts["fc1"])
    np.save(paths["act_fc2"], acts["fc2"])
    np.save(paths["act_fc3"], acts["fc3"])

    config = {
        "dataset": data_source,
        "architecture": "DeepMLP",
        "layers": "784->512->256->10",
        "class": cls,
        "seed": seed,
        "loss": "MaxEnt",
        "lambda_maxent": args.lambda_maxent,
        "pruning": "IMP",
        "pruning_rounds": args.prune_rounds,
        "pruning_rate": args.prune_rate,
        "epochs_per_round": args.epochs_per_round,
        "lr": args.lr,
        "batch_size": args.batch_size,
        "train_subset": args.train_subset,
        "final_sparsity": sparsity,
        "timestamp": utc_timestamp(),
    }
    paths["config"].write_text(json.dumps(config, indent=2), encoding="utf-8")
    metrics = {
        "class": cls,
        "seed": seed,
        "functional": functional,
        "sparsity": sparsity,
        "loss_curves": info.get("loss_curves"),
    }
    paths["metrics"].write_text(json.dumps(metrics, indent=2), encoding="utf-8")


def load_module(out_dir: Path, cls: int, seed: int, device: str) -> Module:
    paths = module_paths(out_dir, cls, seed)
    state = torch.load(paths["checkpoint"], map_location=device)
    mask_t = torch.load(paths["mask"], map_location="cpu")
    metrics = json.loads(paths["metrics"].read_text(encoding="utf-8"))

    weights = {name: state[f"{name}.weight"].cpu().numpy() for name in LAYERS}
    biases = {name: state[f"{name}.bias"].cpu().numpy() for name in LAYERS}
    masks = {name: mask_t[name].cpu().numpy().astype(bool) for name in LAYERS}
    acts = {
        "fc1": np.load(paths["act_fc1"]),
        "fc2": np.load(paths["act_fc2"]),
        "fc3": np.load(paths["act_fc3"]),
    }
    return Module(
        cls=cls, seed=seed, weights=weights, biases=biases, masks=masks, acts=acts,
        functional=metrics["functional"], sparsity=metrics["sparsity"],
    )


def module_from_runtime(
    model: DeepMLP, masks: dict[str, torch.Tensor], cls: int, seed: int,
    functional: dict[str, float], sparsity: dict[str, float], acts: dict[str, np.ndarray],
) -> Module:
    state = model.state_dict()
    weights = {name: state[f"{name}.weight"].cpu().numpy() for name in LAYERS}
    biases = {name: state[f"{name}.bias"].cpu().numpy() for name in LAYERS}
    masks_np = {name: masks[name].cpu().numpy().astype(bool) for name in LAYERS}
    return Module(
        cls=cls, seed=seed, weights=weights, biases=biases, masks=masks_np, acts=acts,
        functional=functional, sparsity=sparsity,
    )

### Low-level metric helpers (numpy) -- mirror metric_sanity_check_model.py

Provides numerical helpers for CKA, Procrustes, correlations, spectra, and output distances.

In [ ]:
def center_columns(x: np.ndarray) -> np.ndarray:
    return x - x.mean(axis=0, keepdims=True)


def linear_cka(x: np.ndarray, y: np.ndarray) -> float:
    x, y = center_columns(x), center_columns(y)
    xty = x.T @ y
    numerator = np.linalg.norm(xty, ord="fro") ** 2
    denominator = np.linalg.norm(x.T @ x, ord="fro") * np.linalg.norm(y.T @ y, ord="fro")
    return float(numerator / (denominator + EPS))


def procrustes_distance(x: np.ndarray, y: np.ndarray) -> float:
    x, y = center_columns(x), center_columns(y)
    x = x / (np.linalg.norm(x, ord="fro") + EPS)
    y = y / (np.linalg.norm(y, ord="fro") + EPS)
    singular_values = np.linalg.svd(x.T @ y, compute_uv=False)
    return float(max(0.0, 1.0 - singular_values.sum()))


def mean_diagonal_abs_corr(x: np.ndarray, y: np.ndarray) -> float:
    """Position-by-position |correlation| averaged over neurons (no matching).

    Units that are dead/constant in either network (common at 96%+ sparsity:
    all incoming weights pruned, only the bias remains) carry no signal and are
    counted as zero correlation -- dividing by their ~0 std would otherwise
    amplify float32 rounding noise into huge bogus values.
    """
    if x.shape[1] != y.shape[1]:
        return float("nan")
    x, y = center_columns(x), center_columns(y)
    x_std = x.std(axis=0)
    y_std = y.std(axis=0)
    alive = (x_std > 1e-6 * max(x_std.max(), EPS)) & (y_std > 1e-6 * max(y_std.max(), EPS))
    if not alive.any():
        return 0.0
    corr = (x[:, alive] * y[:, alive]).mean(axis=0) / (x_std[alive] * y_std[alive])
    corr = np.clip(corr, -1.0, 1.0)
    # Mean over ALL units (dead units contribute 0), keeping the metric comparable
    # across pairs with different numbers of alive units.
    return float(np.sum(np.abs(corr)) / x.shape[1])


def pairwise_abs_corr(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x, y = center_columns(x), center_columns(y)
    x = x / (np.linalg.norm(x, axis=0, keepdims=True) + EPS)
    y = y / (np.linalg.norm(y, axis=0, keepdims=True) + EPS)
    return np.abs(x.T @ y)


def best_matching(similarity: np.ndarray) -> tuple[np.ndarray, float]:
    """Maximise total similarity. Returns (order, mean matched score).

    ``order[i]`` is the column (B index) matched to row i (A index).
    """
    row_ind, col_ind = linear_sum_assignment(-similarity)
    order = np.empty(similarity.shape[0], dtype=int)
    order[row_ind] = col_ind
    return order, float(similarity[row_ind, col_ind].mean())


def participation_ratio(x: np.ndarray) -> float:
    """Effective dimensionality (sum eig)^2 / sum(eig^2) of the covariance."""
    xc = center_columns(x)
    cov = xc.T @ xc
    eig = np.linalg.eigvalsh(cov)
    eig = np.clip(eig, 0, None)
    s1 = eig.sum()
    s2 = (eig ** 2).sum()
    return float((s1 * s1) / (s2 + EPS))


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.astype(bool), b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union else 1.0


def mask_hamming(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.astype(bool), b.astype(bool)
    return float(np.mean(a != b))


def global_iou(masks_a: dict[str, np.ndarray], masks_b: dict[str, np.ndarray]) -> float:
    inter = union = 0
    for name in LAYERS:
        a, b = masks_a[name].astype(bool), masks_b[name].astype(bool)
        inter += int(np.logical_and(a, b).sum())
        union += int(np.logical_or(a, b).sum())
    return float(inter / union) if union else 1.0


def normalized_l2(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + EPS))


def flatten_weights(weights: dict[str, np.ndarray]) -> np.ndarray:
    return np.concatenate([weights[name].ravel() for name in LAYERS])


def mask_spectral_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Normalized L2 distance between singular-value spectra of two 0/1 masks.

    Each layer mask is the biadjacency matrix of a bipartite graph; its singular
    values are invariant under row/column permutations, so this is a structural
    metric that needs NO matching step (and therefore has no matching-inflation
    bias). Caveat: co-spectral masks exist, so equal spectra do not imply equal
    structure -- use as a control alongside permutation-aligned IoU.
    """
    sa = np.linalg.svd(a.astype(np.float64), compute_uv=False)
    sb = np.linalg.svd(b.astype(np.float64), compute_uv=False)
    return float(np.linalg.norm(sa - sb) / (np.linalg.norm(sa) + np.linalg.norm(sb) + EPS))


def iou_null(density_a: float, density_b: float) -> float:
    """Expected IoU of two independent Bernoulli masks with the given densities.

    E[I]/E[U] = dA*dB / (dA + dB - dA*dB): the chance-overlap floor at matched
    density (raw coordinates, no alignment)."""
    denom = density_a + density_b - density_a * density_b
    return float(density_a * density_b / denom) if denom > 0 else 1.0

### Permutation alignment (Git Re-Basin style) for the 784-512-256-10 MLP

Implements hidden-unit permutation alignment so raw coordinate differences do not confound mask/weight comparisons.

In [ ]:
def apply_perm_to_masks(
    masks: dict[str, np.ndarray], order1: np.ndarray, order2: np.ndarray
) -> dict[str, np.ndarray]:
    """Reorder a module's hidden units: fc1 rows by P1, fc2 by (P2 rows, P1 cols),
    fc3 cols by P2.  ``order*[i]`` = source index that lands at position i."""
    return {
        "fc1": masks["fc1"][order1, :],
        "fc2": masks["fc2"][order2, :][:, order1],
        "fc3": masks["fc3"][:, order2],
    }


def apply_perm_to_weights(
    weights: dict[str, np.ndarray], order1: np.ndarray, order2: np.ndarray
) -> dict[str, np.ndarray]:
    return {
        "fc1": weights["fc1"][order1, :],
        "fc2": weights["fc2"][order2, :][:, order1],
        "fc3": weights["fc3"][:, order2],
    }


def apply_perm_to_biases(
    biases: dict[str, np.ndarray], order1: np.ndarray, order2: np.ndarray
) -> dict[str, np.ndarray]:
    return {
        "fc1": biases["fc1"][order1],
        "fc2": biases["fc2"][order2],
        "fc3": biases["fc3"],
    }


def weight_matching(
    a: dict[str, np.ndarray], b: dict[str, np.ndarray], iters: int = 6
) -> tuple[np.ndarray, np.ndarray]:
    """Coordinate-descent weight matching (Ainsworth et al., Git Re-Basin).

    Finds (order1, order2) that align B's hidden units to A using the incoming
    (input/shared) and outgoing (output/shared) connections, iterating between
    the two hidden layers. Works on weights or on binary masks (passed as float).
    Returns orders such that ``B[order]`` is aligned to A.
    """
    a1, a2, a3 = a["fc1"], a["fc2"], a["fc3"]
    b1, b2, b3 = b["fc1"], b["fc2"], b["fc3"]
    h1, h2 = a1.shape[0], a2.shape[0]
    order1 = np.arange(h1)
    order2 = np.arange(h2)
    for _ in range(iters):
        # Update P1 using fc1 incoming (shared input coords) + fc2 outgoing
        # (hidden-2 coords, align B by current order2).
        s1 = a1 @ b1.T + a2.T @ b2[order2, :]
        order1, _ = best_matching(s1)
        # Update P2 using fc2 incoming (hidden-1 coords, align B by order1) +
        # fc3 outgoing (shared output coords).
        s2 = a2 @ b2[:, order1].T + a3.T @ b3
        order2, _ = best_matching(s2)
    return order1, order2


def activation_matching(a: Module, b: Module) -> tuple[np.ndarray, np.ndarray]:
    """Match hidden units by activation correlation on the shared eval inputs.

    Independent per hidden layer (fc1, fc2): activations are functions of the
    same inputs, so no prior-layer alignment is needed. Returns (order1, order2).
    """
    order1, _ = best_matching(pairwise_abs_corr(a.acts["fc1"], b.acts["fc1"]))
    order2, _ = best_matching(pairwise_abs_corr(a.acts["fc2"], b.acts["fc2"]))
    return order1, order2

### Pair metrics

Computes structural, representational, and functional metrics for one pair of modules.

In [ ]:
def output_mse(a: Module, b: Module) -> float:
    return float(np.mean((a.logits - b.logits) ** 2))


def output_kl(a: Module, b: Module) -> float:
    """Mean KL(softmax(A) || softmax(B)) over eval examples."""
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / (e.sum(axis=1, keepdims=True) + EPS)

    pa, pb = softmax(a.logits), softmax(b.logits)
    kl = np.sum(pa * (np.log(pa + EPS) - np.log(pb + EPS)), axis=1)
    return float(np.mean(kl))


def compute_pair_metrics(
    a: Module, b: Module, match_iters: int, eval_labels: np.ndarray | None = None
) -> dict[str, float]:
    """All structural / representational / functional metrics for one pair.

    Keys are flat: ``<metric>__<layer|global>``. If ``eval_labels`` is given,
    CKA/Procrustes are additionally decomposed by input group: examples of the
    pair's rewarded class(es) vs all other (non-rewarded) examples.
    """
    out: dict[str, float] = {}

    # ---- Structural (raw, fixed coordinates) ----
    for name in LAYERS:
        out[f"mask_iou_raw__{name}"] = mask_iou(a.masks[name], b.masks[name])
        out[f"mask_hamming_raw__{name}"] = mask_hamming(a.masks[name], b.masks[name])
        out[f"mask_spectral_dist__{name}"] = mask_spectral_distance(a.masks[name], b.masks[name])
        out[f"mask_iou_null__{name}"] = iou_null(
            float(a.masks[name].mean()), float(b.masks[name].mean())
        )
    out["mask_iou_raw__global"] = global_iou(a.masks, b.masks)
    out["mask_spectral_dist__global"] = float(
        np.mean([out[f"mask_spectral_dist__{name}"] for name in LAYERS])
    )
    out["mask_iou_null__global"] = iou_null(
        float(sum(a.masks[n].sum() for n in LAYERS) / sum(a.masks[n].size for n in LAYERS)),
        float(sum(b.masks[n].sum() for n in LAYERS) / sum(b.masks[n].size for n in LAYERS)),
    )

    # ---- Structural (permutation aligned) ----
    o1_w, o2_w = weight_matching(
        {k: a.masks[k].astype(float) for k in LAYERS},
        {k: b.masks[k].astype(float) for k in LAYERS},
        iters=match_iters,
    )
    b_masks_wm = apply_perm_to_masks(b.masks, o1_w, o2_w)
    o1_a, o2_a = activation_matching(a, b)
    b_masks_am = apply_perm_to_masks(b.masks, o1_a, o2_a)
    for name in LAYERS:
        out[f"mask_iou_perm_maskmatch__{name}"] = mask_iou(a.masks[name], b_masks_wm[name])
        out[f"mask_iou_perm_actmatch__{name}"] = mask_iou(a.masks[name], b_masks_am[name])
    out["mask_iou_perm_maskmatch__global"] = global_iou(a.masks, b_masks_wm)
    out["mask_iou_perm_actmatch__global"] = global_iou(a.masks, b_masks_am)

    # ---- Optional weight comparisons ----
    out["weight_cosine_raw__global"] = float(
        np.dot(flatten_weights(a.weights), flatten_weights(b.weights))
        / ((np.linalg.norm(flatten_weights(a.weights)) * np.linalg.norm(flatten_weights(b.weights))) + EPS)
    )
    out["weight_l2_raw__global"] = normalized_l2(
        flatten_weights(a.weights), flatten_weights(b.weights)
    )
    b_weights_wm = apply_perm_to_weights(b.weights, o1_w, o2_w)
    out["weight_l2_perm__global"] = normalized_l2(
        flatten_weights(a.weights), flatten_weights(b_weights_wm)
    )

    # ---- Representational ----
    for name in LAYERS:
        xa, xb = a.acts[name], b.acts[name]
        out[f"cka_linear__{name}"] = linear_cka(xa, xb)
        out[f"procrustes_distance__{name}"] = procrustes_distance(xa, xb)
        out[f"activation_corr_raw__{name}"] = mean_diagonal_abs_corr(xa, xb)
        _, matched = best_matching(pairwise_abs_corr(xa, xb))
        out[f"activation_corr_matched__{name}"] = matched
        out[f"participation_ratio_a__{name}"] = participation_ratio(xa)
        out[f"participation_ratio_b__{name}"] = participation_ratio(xb)

    # ---- Representational, decomposed by input group (rewarded vs non-rewarded).
    # Tests whether global similarity is driven by the MaxEnt "flat" behaviour on
    # non-rewarded classes rather than by the rewarded-class representation
    # (pattern D in setting_esatto_test_sperimentali_moduli.md, sec. 9).
    if eval_labels is not None:
        rewarded_classes = {c for c in (a.cls, b.cls) if c >= 0}
        if rewarded_classes:
            rew_idx = np.isin(eval_labels, sorted(rewarded_classes))
            min_n = 20  # below this, subset CKA/Procrustes is too noisy
            for name in LAYERS:
                xa, xb = a.acts[name], b.acts[name]
                for group, idx in (("rew", rew_idx), ("nonrew", ~rew_idx)):
                    if idx.sum() >= min_n:
                        out[f"cka_linear_{group}__{name}"] = linear_cka(xa[idx], xb[idx])
                        out[f"procrustes_{group}__{name}"] = procrustes_distance(xa[idx], xb[idx])
                    else:
                        out[f"cka_linear_{group}__{name}"] = float("nan")
                        out[f"procrustes_{group}__{name}"] = float("nan")
            out["n_rew_examples__global"] = float(rew_idx.sum())

    # ---- Functional (pairwise) ----
    out["output_mse__global"] = output_mse(a, b)
    out["output_kl__global"] = output_kl(a, b)
    return out

### Control-module construction (permuted / random)

Builds known-answer controls: identical, permuted copy, and density-matched random baseline.

In [ ]:
def make_permuted_control(module: Module, seed: int) -> Module:
    """Return a hidden-unit permutation of ``module`` (same function, new coords)."""
    rng = np.random.default_rng(seed)
    h1 = module.weights["fc1"].shape[0]
    h2 = module.weights["fc2"].shape[0]
    o1 = rng.permutation(h1)
    o2 = rng.permutation(h2)
    weights = apply_perm_to_weights(module.weights, o1, o2)
    biases = apply_perm_to_biases(module.biases, o1, o2)
    masks = apply_perm_to_masks(module.masks, o1, o2)
    acts = {
        "fc1": module.acts["fc1"][:, o1],
        "fc2": module.acts["fc2"][:, o2],
        "fc3": module.acts["fc3"],  # output unchanged by hidden permutation
    }
    return Module(
        cls=module.cls, seed=module.seed, weights=weights, biases=biases, masks=masks,
        acts=acts, functional=module.functional, sparsity=module.sparsity,
    )


def make_random_baseline(module: Module, eval_x: torch.Tensor, seed: int, device: str) -> Module:
    """Random sparse module with the same per-layer density as ``module``."""
    seed_everything(seed)
    model = DeepMLP().to(device)
    masks: dict[str, torch.Tensor] = {}
    rng = np.random.default_rng(seed)
    for name in LAYERS:
        density = float(module.masks[name].mean())
        m = (rng.random(module.masks[name].shape) < density).astype(np.float32)
        masks[name] = torch.from_numpy(m).to(device)
    apply_masks(model, masks)
    acts = model.collect_activations(eval_x.to(device))
    return module_from_runtime(
        model, masks, cls=-1, seed=seed,
        functional={"rewarded_accuracy": float("nan"), "nonrewarded_entropy": float("nan")},
        sparsity=layer_sparsity({k: v for k, v in masks.items()}), acts=acts,
    )

### Pair enumeration

Creates the experimental pair categories used in the report: same-class/diff-seed and cross-class baselines.

In [ ]:
@dataclass
class PairSpec:
    a: Module
    b: Module
    category: str
    class_a: int
    class_b: int
    seed_a: int
    seed_b: int


def categorize(a: Module, b: Module) -> str | None:
    if a.cls == b.cls and a.seed != b.seed:
        return "same_class_diff_seed"
    if a.cls != b.cls and a.seed == b.seed:
        return "diff_class_same_seed"
    if a.cls != b.cls and a.seed != b.seed:
        return "diff_class_diff_seed"
    return None  # same class & same seed -> only the identical control covers it


def build_pairs(modules: list[Module], eval_x: torch.Tensor, args, device: str) -> list[PairSpec]:
    pairs: list[PairSpec] = []

    # Controls: one per module. These are "known answer" comparisons:
    # - identical must be perfect;
    # - permuted_control is the same network in different hidden-unit coordinates,
    #   so invariant/aligned metrics must recover it while raw coordinate metrics
    #   are allowed to fail;
    # - random_baseline has matched sparsity but unrelated weights/activations.
    # The experimental same/different-class patterns are interpreted only after
    # these controls pass.
    for m in modules:
        pairs.append(PairSpec(m, m, "identical", m.cls, m.cls, m.seed, m.seed))
        perm = make_permuted_control(m, seed=10_000 + m.cls * 100 + m.seed)
        pairs.append(PairSpec(m, perm, "permuted_control", m.cls, m.cls, m.seed, m.seed))
        rand = make_random_baseline(m, eval_x, seed=20_000 + m.cls * 100 + m.seed, device=device)
        pairs.append(PairSpec(m, rand, "random_baseline", m.cls, rand.cls, m.seed, rand.seed))

    # Experimental comparisons: all unordered pairs of distinct real modules.
    # The pair category separates "same class across seeds" from two cross-class
    # baselines, one holding the seed fixed and one varying both seed and class.
    for a, b in combinations(modules, 2):
        cat = categorize(a, b)
        if cat is not None:
            pairs.append(PairSpec(a, b, cat, a.cls, b.cls, a.seed, b.seed))
    return pairs

### Composition (optional)

Evaluates all-class weight-sum composition for seeds that have a complete class set.

In [ ]:
@torch.no_grad()
def evaluate_composition(
    modules: list[Module], classes: list[int], eval_x: torch.Tensor, eval_y: torch.Tensor,
    device: str,
) -> dict[str, Any]:
    """Compose per-seed modules by weight summation and measure accuracy/entropy.

    For each seed for which all requested classes have a module, build a single
    network whose weights are the sum of the masked module weights, then evaluate
    top-1 accuracy (restricted to the trained classes) and mean entropy.
    """
    by_seed: dict[int, dict[int, Module]] = {}
    for m in modules:
        by_seed.setdefault(m.seed, {})[m.cls] = m

    results: dict[str, Any] = {}
    class_set = set(classes)
    eval_x_dev = eval_x.to(device)
    keep = torch.tensor([y in class_set for y in eval_y.tolist()])
    for seed, by_class in sorted(by_seed.items()):
        if not class_set.issubset(by_class.keys()):
            continue
        composed = DeepMLP().to(device)
        with torch.no_grad():
            for name in LAYERS:
                w = sum(
                    torch.from_numpy(by_class[c].weights[name] * by_class[c].masks[name]).float()
                    for c in classes
                )
                bsum = sum(torch.from_numpy(by_class[c].biases[name]).float() for c in classes)
                getattr(composed, name).weight.copy_(w.to(device))
                # theta_merged = sum_i theta_i (Caso et al. sec. 3.3.1): biases are
                # parameters too, so they are summed rather than zeroed.
                getattr(composed, name).bias.copy_(bsum.to(device))
        logits = composed(eval_x_dev).cpu()
        probs = F.softmax(logits, dim=1)
        entropy = -(probs * torch.log(probs + EPS)).sum(dim=1)
        # Restrict logits to trained classes for the argmax decision.
        class_idx = torch.tensor(sorted(classes))
        restricted = logits[:, class_idx]
        preds = class_idx[restricted.argmax(dim=1)]
        acc = float((preds[keep] == eval_y[keep]).float().mean()) if keep.any() else float("nan")
        results[str(seed)] = {
            "composition_accuracy": acc,
            "composition_entropy": float(entropy.mean()),
            "n_eval": int(keep.sum()),
        }
    return results

### Pairwise composition & interpolation barrier

Evaluates pairwise weight-sum composition and optional interpolation barriers.

In [ ]:
def _set_model_params(
    model: DeepMLP, weights: dict[str, np.ndarray], biases: dict[str, np.ndarray]
) -> None:
    with torch.no_grad():
        for name in LAYERS:
            layer = getattr(model, name)
            layer.weight.copy_(torch.from_numpy(np.ascontiguousarray(weights[name])).float())
            layer.bias.copy_(torch.from_numpy(np.ascontiguousarray(biases[name])).float())


@torch.no_grad()
def pairwise_composition_metrics(
    a: Module, b: Module, eval_x: torch.Tensor, eval_y: torch.Tensor, device: str
) -> dict[str, float]:
    """Merge the two modules (theta_merged = theta_a + theta_b, Caso et al. 3.3.1
    'pairwise merges') and measure how well the merge serves both rewarded classes.

    - composition_accuracy: unrestricted-argmax accuracy on examples of the pair's
      rewarded class(es);
    - composition_interference: mean over the two modules of
      (solo rewarded accuracy - merged accuracy on that module's class);
    - composition_entropy_nonrew: mean prediction entropy on all other examples
      (should stay near ln(10) if the merge preserves MaxEnt behaviour).
    """
    weights = {n: a.weights[n] * a.masks[n] + b.weights[n] * b.masks[n] for n in LAYERS}
    biases = {n: a.biases[n] + b.biases[n] for n in LAYERS}
    model = DeepMLP().to(device)
    _set_model_params(model, weights, biases)
    model.eval()
    logits = model(eval_x.to(device)).cpu()
    probs = F.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + EPS)).sum(dim=1)
    preds = logits.argmax(dim=1)

    rewarded = sorted({c for c in (a.cls, b.cls) if c >= 0})
    rew_mask = torch.isin(eval_y, torch.tensor(rewarded))
    acc = float((preds[rew_mask] == eval_y[rew_mask]).float().mean()) if rew_mask.any() else float("nan")

    interferences = []
    for m in (a, b):
        solo = m.functional.get("rewarded_accuracy")
        cls_mask = eval_y == m.cls
        if solo is None or np.isnan(solo) or not cls_mask.any():
            continue
        merged_acc = float((preds[cls_mask] == m.cls).float().mean())
        interferences.append(solo - merged_acc)
    nonrew = ~rew_mask
    return {
        "composition_accuracy__global": acc,
        "composition_interference__global": float(np.mean(interferences)) if interferences else float("nan"),
        "composition_entropy_nonrew__global": float(entropy[nonrew].mean()) if nonrew.any() else float("nan"),
    }


@torch.no_grad()
def interpolation_barrier(
    a: Module, b: Module, eval_x: torch.Tensor, eval_y: torch.Tensor,
    lam: float, device: str, match_iters: int,
    alphas: tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0),
) -> dict[str, Any]:
    """MaxEnt-loss / rewarded-accuracy barrier along the linear interpolation
    between two same-class modules, before (raw) and after (aligned) Git
    Re-Basin weight matching (Entezari et al. 2022; Ainsworth et al. 2023).

    barrier_loss = max_alpha [ L(alpha) - ((1-alpha) L(0) + alpha L(1)) ].
    Caveats (report as descriptive evidence): supports differ, so the midpoint
    lives on the union support with halved disjoint weights; no REPAIR-style
    renormalization is applied.
    """
    rewarded = a.cls
    o1, o2 = weight_matching(a.weights, b.weights, iters=match_iters)
    variants = {
        "raw": (b.weights, b.biases),
        "aligned": (apply_perm_to_weights(b.weights, o1, o2), apply_perm_to_biases(b.biases, o1, o2)),
    }
    model = DeepMLP().to(device)
    eval_x_dev = eval_x.to(device)
    eval_y_dev = eval_y.to(device)
    result: dict[str, Any] = {"alphas": list(alphas)}
    for tag, (bw, bb) in variants.items():
        losses, accs = [], []
        for alpha in alphas:
            weights = {n: (1 - alpha) * a.weights[n] + alpha * bw[n] for n in LAYERS}
            biases = {n: (1 - alpha) * a.biases[n] + alpha * bb[n] for n in LAYERS}
            _set_model_params(model, weights, biases)
            model.eval()
            logits = model(eval_x_dev)
            losses.append(float(maxent_loss(logits, eval_y_dev, rewarded, lam).item()))
            rew_mask = eval_y_dev == rewarded
            accs.append(float((logits.argmax(dim=1)[rew_mask] == rewarded).float().mean())
                        if rew_mask.any() else float("nan"))
        losses_np, accs_np = np.array(losses), np.array(accs)
        lin_loss = (1 - np.array(alphas)) * losses_np[0] + np.array(alphas) * losses_np[-1]
        lin_acc = (1 - np.array(alphas)) * accs_np[0] + np.array(alphas) * accs_np[-1]
        result[f"loss_curve_{tag}"] = losses
        result[f"acc_curve_{tag}"] = accs
        result[f"barrier_loss_{tag}"] = float(np.max(losses_np - lin_loss))
        result[f"barrier_accdrop_{tag}"] = float(np.max(lin_acc - accs_np))
    return result

### Aggregation

Aggregates pair-level metrics into mean, standard deviation, min, and max tables.

In [ ]:
def aggregate(pair_records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Aggregate every metric by (pair_category, metric, layer)."""
    # Collect metric keys (metric__layer) across all records.
    metric_keys: set[str] = set()
    for rec in pair_records:
        metric_keys.update(k for k in rec["metrics"].keys())

    rows: list[dict[str, Any]] = []
    for category in PAIR_CATEGORIES:
        cat_records = [r for r in pair_records if r["category"] == category]
        if not cat_records:
            continue
        for full_key in sorted(metric_keys):
            metric, _, layer = full_key.partition("__")
            values = np.array(
                [r["metrics"][full_key] for r in cat_records if full_key in r["metrics"]],
                dtype=float,
            )
            values = values[~np.isnan(values)]
            if values.size == 0:
                continue
            rows.append({
                "pair_category": category,
                "metric": metric,
                "layer": layer or "global",
                "n_pairs": int(values.size),
                "mean": float(values.mean()),
                "std": float(values.std(ddof=1)) if values.size > 1 else 0.0,
                "min": float(values.min()),
                "max": float(values.max()),
            })
    return rows


def class_pair_matrix(
    pair_records: list[dict[str, Any]], metric_key: str, classes: list[int]
) -> np.ndarray:
    """Mean of ``metric_key`` for each (class_a, class_b), averaged over seeds.

    Uses experimental categories only (same/diff class). Symmetric matrix.
    """
    n = len(classes)
    idx = {c: i for i, c in enumerate(classes)}
    acc = np.full((n, n), np.nan)
    counts = np.zeros((n, n))
    sums = np.zeros((n, n))
    exp_cats = {"same_class_diff_seed", "diff_class_same_seed", "diff_class_diff_seed"}
    for r in pair_records:
        if r["category"] not in exp_cats:
            continue
        ca, cb = r["class_a"], r["class_b"]
        if ca not in idx or cb not in idx:
            continue
        v = r["metrics"].get(metric_key)
        if v is None or np.isnan(v):
            continue
        for (i, j) in ((idx[ca], idx[cb]), (idx[cb], idx[ca])):
            sums[i, j] += v
            counts[i, j] += 1
    with np.errstate(invalid="ignore", divide="ignore"):
        acc = np.where(counts > 0, sums / counts, np.nan)
    return acc

### Significance: same_class_diff_seed vs diff_class_diff_seed

Runs same-class versus cross-class tests, including a module-label permutation test.

In [ ]:
def significance_tests(
    pair_records: list[dict[str, Any]], n_perm: int = 1000, rng_seed: int = 0
) -> list[dict[str, Any]]:
    """Test, per metric, whether same-class pairs differ from cross-class pairs.

    Both groups are restricted to seed_a != seed_b so seed structure is matched.
    Reports Mann-Whitney U + rank-biserial effect size (pair-level; p-values are
    descriptive because pairs share modules) and a module-label permutation test:
    class labels are shuffled across modules and the group mean difference is
    recomputed from the SAME pair metrics, which respects the dependence between
    pairs sharing a module.
    """
    from scipy.stats import mannwhitneyu

    diffseed = [r for r in pair_records
                if r["category"] in ("same_class_diff_seed", "diff_class_diff_seed")]
    if not diffseed:
        return []
    module_ids = sorted({(r["class_a"], r["seed_a"]) for r in diffseed}
                        | {(r["class_b"], r["seed_b"]) for r in diffseed})
    labels = np.array([cls for cls, _ in module_ids])
    mod_index = {m: i for i, m in enumerate(module_ids)}
    pair_idx = np.array([
        (mod_index[(r["class_a"], r["seed_a"])], mod_index[(r["class_b"], r["seed_b"])])
        for r in diffseed
    ])

    metric_keys = sorted({k for r in diffseed for k in r["metrics"]
                          if not k.startswith(("participation_ratio", "n_rew_examples",
                                               "mask_iou_null"))})
    rng = np.random.default_rng(rng_seed)
    # The permutation test shuffles class labels attached to whole modules, not
    # pair labels. This is the important dependence correction: many pairwise
    # rows share the same trained module, so treating pairs as independent would
    # make p-values look stronger than they are.
    perms = [rng.permutation(len(labels)) for _ in range(n_perm)]

    rows: list[dict[str, Any]] = []
    for key in metric_keys:
        values = np.array([r["metrics"].get(key, np.nan) for r in diffseed], dtype=float)
        valid = ~np.isnan(values)
        if valid.sum() < 4:
            continue
        same_obs = labels[pair_idx[:, 0]] == labels[pair_idx[:, 1]]
        v, s = values[valid], same_obs[valid]
        if s.sum() == 0 or (~s).sum() == 0:
            continue
        delta = float(v[s].mean() - v[~s].mean())
        try:
            u_stat, p_val = mannwhitneyu(v[s], v[~s], alternative="two-sided")
            # rank-biserial in [-1, 1]; positive = same-class values are larger.
            rank_biserial = float(2.0 * u_stat / (s.sum() * (~s).sum()) - 1.0)
        except ValueError:
            u_stat, p_val, rank_biserial = np.nan, np.nan, np.nan
        # Module-label permutation test on the same pair values.
        count = 0
        n_used = 0
        for perm in perms:
            plabels = labels[perm]
            s_p = (plabels[pair_idx[:, 0]] == plabels[pair_idx[:, 1]])[valid]
            if s_p.sum() == 0 or (~s_p).sum() == 0:
                continue
            n_used += 1
            if abs(v[s_p].mean() - v[~s_p].mean()) >= abs(delta):
                count += 1
        perm_p = float((count + 1) / (n_used + 1)) if n_used else float("nan")
        metric, _, layer = key.partition("__")
        rows.append({
            "metric": metric, "layer": layer or "global",
            "n_same": int(s.sum()), "n_diff": int((~s).sum()),
            "mean_same": float(v[s].mean()), "mean_diff": float(v[~s].mean()),
            "delta": delta,
            "mannwhitney_u": float(u_stat), "mannwhitney_p": float(p_val),
            "rank_biserial": rank_biserial, "perm_p": perm_p,
        })
    return rows

### Plots

Creates the figures shown in the final report and saved under the output folder.

In [ ]:
def _import_plt():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def plot_structural_by_category(rows, fig_dir: Path, pair_records) -> None:
    plt = _import_plt()
    metric = "mask_iou_perm_maskmatch"
    cats = [c for c in PAIR_CATEGORIES if any(
        r["category"] == c and metric + "__global" in r["metrics"] for r in pair_records)]
    data = [
        [r["metrics"][metric + "__global"] for r in pair_records
         if r["category"] == c and not np.isnan(r["metrics"].get(metric + "__global", np.nan))]
        for c in cats
    ]
    fig, ax = plt.subplots(figsize=(10, 5))
    if any(len(d) for d in data):
        ax.boxplot([d if d else [np.nan] for d in data], tick_labels=cats, showmeans=True)
    nulls = [r["metrics"].get("mask_iou_null__global", np.nan) for r in pair_records
             if r["category"] in ("same_class_diff_seed", "diff_class_diff_seed")]
    nulls = [n for n in nulls if not np.isnan(n)]
    if nulls:
        ax.axhline(float(np.mean(nulls)), color="gray", linestyle="--", linewidth=1,
                   label=f"analytic IoU null (indep. masks) = {np.mean(nulls):.3f}")
        ax.legend(fontsize=8)
    ax.set_title("Permutation-aligned mask IoU (global) by pair category")
    ax.set_ylabel("aligned IoU (mask matching)")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(fig_dir / "structural_metrics_by_pair_type.png", dpi=160)
    plt.close(fig)


def plot_representation_by_category(fig_dir: Path, pair_records) -> None:
    plt = _import_plt()
    for metric, fname, ylabel in [
        ("cka_linear", "representation_metrics_by_pair_type.png", "linear CKA (higher=similar)"),
        ("procrustes_distance", "procrustes_by_pair_type.png", "Procrustes distance (lower=similar)"),
    ]:
        cats = [c for c in PAIR_CATEGORIES if any(r["category"] == c for r in pair_records)]
        fig, axes = plt.subplots(1, len(LAYERS), figsize=(15, 5), sharey=True)
        for ax, layer in zip(axes, LAYERS):
            data = [
                [r["metrics"][f"{metric}__{layer}"] for r in pair_records
                 if r["category"] == c and not np.isnan(r["metrics"].get(f"{metric}__{layer}", np.nan))]
                for c in cats
            ]
            ax.boxplot([d if d else [np.nan] for d in data], tick_labels=cats, showmeans=True)
            ax.set_title(f"{metric} - {layer}")
            ax.tick_params(axis="x", rotation=35)
            ax.grid(axis="y", alpha=0.3)
        axes[0].set_ylabel(ylabel)
        fig.tight_layout()
        fig.savefig(fig_dir / fname, dpi=160)
        plt.close(fig)


def plot_class_heatmaps(fig_dir: Path, pair_records, classes: list[int]) -> None:
    """Class x class mean heatmaps (averaged over seeds) for the three levels:
    CKA (fc2), Procrustes (fc2) and permutation-aligned IoU (global)."""
    plt = _import_plt()
    specs = [
        ("cka_linear__fc2", "Mean linear CKA (fc2)", "class_class_cka_heatmap.png"),
        ("procrustes_distance__fc2", "Mean Procrustes distance (fc2)",
         "class_class_procrustes_heatmap.png"),
        ("mask_iou_perm_maskmatch__global", "Mean aligned mask IoU (global)",
         "class_class_aligned_iou_heatmap.png"),
    ]
    for metric_key, title, fname in specs:
        mat = class_pair_matrix(pair_records, metric_key, classes)
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(mat, cmap="viridis",
                       vmin=np.nanmin(mat) if np.isfinite(mat).any() else 0,
                       vmax=np.nanmax(mat) if np.isfinite(mat).any() else 1)
        ax.set_xticks(range(len(classes)), classes)
        ax.set_yticks(range(len(classes)), classes)
        ax.set_xlabel("class b")
        ax.set_ylabel("class a")
        ax.set_title(f"{title}, averaged over seeds (diagonal = same class)")
        for i in range(len(classes)):
            for j in range(len(classes)):
                if np.isfinite(mat[i, j]):
                    ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                            color="white", fontsize=7)
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        fig.savefig(fig_dir / fname, dpi=160)
        plt.close(fig)


def plot_structure_vs_representation(fig_dir: Path, pair_records) -> None:
    plt = _import_plt()
    colors = {
        "identical": "#000000", "permuted_control": "#9467bd", "random_baseline": "#7f7f7f",
        "same_class_diff_seed": "#2ca02c", "diff_class_same_seed": "#ff7f0e",
        "diff_class_diff_seed": "#d62728",
    }
    fig, ax = plt.subplots(figsize=(8, 6))
    for category in PAIR_CATEGORIES:
        xs, ys = [], []
        for r in pair_records:
            if r["category"] != category:
                continue
            x = r["metrics"].get("mask_iou_perm_maskmatch__global", np.nan)
            y = r["metrics"].get("cka_linear__fc2", np.nan)
            if not (np.isnan(x) or np.isnan(y)):
                xs.append(x)
                ys.append(y)
        if xs:
            ax.scatter(xs, ys, label=category, alpha=0.6, color=colors.get(category), s=25)
    ax.set_xlabel("permutation-aligned mask IoU (global)")
    ax.set_ylabel("linear CKA (fc2)")
    ax.set_title("Structure vs representation")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(fig_dir / "structure_vs_representation_scatter.png", dpi=160)
    plt.close(fig)


def plot_composition_vs_similarity(fig_dir: Path, pair_records) -> None:
    """Does similarity predict pairwise composability? One panel per metric,
    y = pairwise merge accuracy, Spearman rho annotated."""
    plt = _import_plt()
    from scipy.stats import spearmanr

    metric_specs = [
        ("mask_iou_perm_maskmatch__global", "aligned mask IoU (global)"),
        ("cka_linear__fc2", "linear CKA (fc2)"),
        ("procrustes_distance__fc2", "Procrustes distance (fc2)"),
        ("output_kl__global", "output KL"),
    ]
    colors = {
        "same_class_diff_seed": "#2ca02c", "diff_class_same_seed": "#ff7f0e",
        "diff_class_diff_seed": "#d62728",
    }
    recs = [r for r in pair_records
            if r["category"] in colors and "composition_accuracy__global" in r["metrics"]]
    if not recs:
        return
    fig, axes = plt.subplots(1, len(metric_specs), figsize=(5 * len(metric_specs), 4.5),
                             sharey=True)
    for ax, (key, label) in zip(np.atleast_1d(axes), metric_specs):
        xs_all, ys_all = [], []
        for cat, color in colors.items():
            xs = [r["metrics"].get(key, np.nan) for r in recs if r["category"] == cat]
            ys = [r["metrics"].get("composition_accuracy__global", np.nan)
                  for r in recs if r["category"] == cat]
            pts = [(x, y) for x, y in zip(xs, ys) if not (np.isnan(x) or np.isnan(y))]
            if pts:
                ax.scatter(*zip(*pts), label=cat, alpha=0.6, color=color, s=25)
                xs_all += [p[0] for p in pts]
                ys_all += [p[1] for p in pts]
        if len(xs_all) >= 3 and np.std(xs_all) > 0 and np.std(ys_all) > 0:
            rho, p = spearmanr(xs_all, ys_all)
            ax.set_title(f"{label}\nSpearman rho={rho:.2f} (p={p:.3f})", fontsize=9)
        else:
            ax.set_title(label, fontsize=9)
        ax.set_xlabel(label, fontsize=8)
        ax.grid(alpha=0.3)
    np.atleast_1d(axes)[0].set_ylabel("pairwise composition accuracy")
    np.atleast_1d(axes)[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(fig_dir / "composition_vs_similarity.png", dpi=160)
    plt.close(fig)


def plot_barrier_curves(fig_dir: Path, pair_records) -> None:
    """Mean MaxEnt-loss interpolation curves, raw vs weight-matched (aligned),
    for same-class different-seed pairs; permuted control as reference."""
    plt = _import_plt()
    groups = {"same_class_diff_seed": "-", "permuted_control": ":"}
    fig, ax = plt.subplots(figsize=(8, 5))
    plotted = False
    for cat, style in groups.items():
        recs = [r for r in pair_records if r["category"] == cat and r.get("barrier")]
        if not recs:
            continue
        alphas = recs[0]["barrier"]["alphas"]
        for tag, color in (("raw", "#d62728"), ("aligned", "#2ca02c")):
            curves = np.array([r["barrier"][f"loss_curve_{tag}"] for r in recs])
            ax.plot(alphas, curves.mean(axis=0), style, color=color,
                    label=f"{cat} - {tag} (n={len(recs)})")
            plotted = True
    if not plotted:
        plt.close(fig)
        return
    ax.set_xlabel("interpolation alpha")
    ax.set_ylabel("MaxEnt loss on eval set")
    ax.set_title("Interpolation barrier: raw vs permutation-aligned (weight matching)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(fig_dir / "interpolation_barrier.png", dpi=160)
    plt.close(fig)


def plot_rewarded_vs_nonrewarded(fig_dir: Path, pair_records) -> None:
    """Group-decomposed CKA for same-class pairs: rewarded vs non-rewarded inputs
    (tests whether global similarity is driven by the MaxEnt flat behaviour)."""
    plt = _import_plt()
    recs = [r for r in pair_records if r["category"] == "same_class_diff_seed"]
    if not recs:
        return
    fig, axes = plt.subplots(1, len(LAYERS), figsize=(13, 4.5), sharey=True)
    for ax, layer in zip(axes, LAYERS):
        data, labels = [], []
        for group in ("rew", "nonrew"):
            vals = [r["metrics"].get(f"cka_linear_{group}__{layer}", np.nan) for r in recs]
            vals = [v for v in vals if not np.isnan(v)]
            if vals:
                data.append(vals)
                labels.append(f"{group} (n={len(vals)})")
        if data:
            ax.boxplot(data, tick_labels=labels, showmeans=True)
        ax.set_title(f"CKA by input group - {layer}")
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("linear CKA (same-class diff-seed pairs)")
    fig.tight_layout()
    fig.savefig(fig_dir / "rewarded_vs_nonrewarded_cka.png", dpi=160)
    plt.close(fig)


def make_all_plots(out_dir: Path, rows, pair_records, classes) -> list[str]:
    fig_dir = out_dir / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    made = []
    try:
        plot_structural_by_category(rows, fig_dir, pair_records)
        plot_representation_by_category(fig_dir, pair_records)
        plot_class_heatmaps(fig_dir, pair_records, classes)
        plot_structure_vs_representation(fig_dir, pair_records)
        plot_composition_vs_similarity(fig_dir, pair_records)
        plot_barrier_curves(fig_dir, pair_records)
        plot_rewarded_vs_nonrewarded(fig_dir, pair_records)
        made = sorted(p.name for p in fig_dir.glob("*.png"))
    except Exception as exc:  # plotting must never crash the pipeline
        print(f"WARNING: plotting failed: {exc}", file=sys.stderr)
    return made

### Reports

Writes the concise report and the full methodological report.

In [ ]:
def _mean_for(rows, category, metric, layer) -> float | None:
    for r in rows:
        if r["pair_category"] == category and r["metric"] == metric and r["layer"] == layer:
            return r["mean"]
    return None


def signal_line(rows, metric, layer, higher_is_more_similar: bool) -> str:
    same = _mean_for(rows, "same_class_diff_seed", metric, layer)
    cross = _mean_for(rows, "diff_class_diff_seed", metric, layer)
    if same is None or cross is None:
        return f"- {metric} ({layer}): insufficient data."
    if higher_is_more_similar:
        delta = same - cross
        verdict = "signal" if delta > 0.01 else ("weak/none" if abs(delta) <= 0.01 else "inverted")
        return (f"- {metric} ({layer}): same-class={same:.3f} vs cross-class={cross:.3f} "
                f"(Δ={delta:+.3f}) -> {verdict}")
    delta = cross - same  # for distances, same-class should be smaller
    verdict = "signal" if delta > 0.01 else ("weak/none" if abs(delta) <= 0.01 else "inverted")
    return (f"- {metric} ({layer}): same-class={same:.3f} vs cross-class={cross:.3f} "
            f"(Δ_dist={ -delta:+.3f}) -> {verdict}")


def _spearman_line(pair_records, metric_key: str, label: str) -> str | None:
    """Spearman correlation of a similarity metric with pairwise merge accuracy."""
    from scipy.stats import spearmanr

    cats = ("same_class_diff_seed", "diff_class_same_seed", "diff_class_diff_seed")
    pts = [
        (r["metrics"].get(metric_key, np.nan),
         r["metrics"].get("composition_accuracy__global", np.nan))
        for r in pair_records if r["category"] in cats
    ]
    pts = [(x, y) for x, y in pts if not (np.isnan(x) or np.isnan(y))]
    if len(pts) < 3:
        return None
    xs, ys = zip(*pts)
    if np.std(xs) == 0 or np.std(ys) == 0:
        return f"- {label} vs merge accuracy: degenerate (constant values, n={len(pts)})"
    rho, p = spearmanr(xs, ys)
    return f"- {label} vs merge accuracy: rho={rho:+.2f} (p={p:.3f}, n={len(pts)})"


def write_reports(
    out_dir: Path, rows, pair_records, modules, classes, seeds, args, data_source,
    composition, figures, elapsed, sig_rows=None,
) -> None:
    n_modules = len(modules)
    n_pairs = len(pair_records)
    by_cat = {c: sum(1 for r in pair_records if r["category"] == c) for c in PAIR_CATEGORIES}

    # ---- concise report ----
    concise = [
        "# Calibrated MaxEnt module metrics - concise report",
        "",
        f"- Generated: {utc_timestamp()}",
        f"- Data: {data_source}; classes={classes}; seeds={seeds}",
        f"- Modules: {n_modules}; pairs: {n_pairs} {by_cat}",
        f"- Pruning: IMP {args.prune_rounds} rounds x {args.prune_rate:.0%}, "
        f"epochs/round={args.epochs_per_round}, lambda={args.lambda_maxent}",
        "",
        "## Sanity controls (must hold)",
    ]
    for metric, layer, hi in [
        ("mask_iou_raw", "global", True),
        ("mask_iou_perm_maskmatch", "global", True),
        ("cka_linear", "fc2", True),
        ("output_mse", "global", False),
    ]:
        ident = _mean_for(rows, "identical", metric, layer)
        perm = _mean_for(rows, "permuted_control", metric, layer)
        rand = _mean_for(rows, "random_baseline", metric, layer)
        concise.append(
            f"- {metric} ({layer}): identical={_fmt(ident)}, permuted={_fmt(perm)}, "
            f"random={_fmt(rand)}"
        )
    concise += [
        "",
        "Expected: identical is perfect; permuted_control has LOW raw IoU but HIGH "
        "aligned IoU / CKA and ~0 output MSE (the permutation calibration); "
        "random_baseline is low on every similarity.",
        "",
        "## Structural signal (same-class vs cross-class)",
        signal_line(rows, "mask_iou_perm_maskmatch", "global", True),
        signal_line(rows, "mask_iou_perm_actmatch", "global", True),
        signal_line(rows, "mask_iou_raw", "global", True),
        signal_line(rows, "mask_spectral_dist", "global", False),
        "",
        "## Representational signal (same-class vs cross-class)",
        signal_line(rows, "cka_linear", "fc1", True),
        signal_line(rows, "cka_linear", "fc2", True),
        signal_line(rows, "cka_linear", "fc3", True),
        signal_line(rows, "procrustes_distance", "fc1", False),
        signal_line(rows, "procrustes_distance", "fc2", False),
        signal_line(rows, "procrustes_distance", "fc3", False),
        "",
        "## Rewarded vs non-rewarded (same-class pairs, CKA fc2/fc3)",
    ]
    for layer in ("fc2", "fc3"):
        rew = _mean_for(rows, "same_class_diff_seed", "cka_linear_rew", layer)
        nonrew = _mean_for(rows, "same_class_diff_seed", "cka_linear_nonrew", layer)
        if rew is not None and nonrew is not None:
            concise.append(
                f"- {layer}: rewarded={rew:.3f} vs non-rewarded={nonrew:.3f} "
                f"({'non-rewarded MORE similar' if nonrew > rew else 'rewarded MORE similar'})"
            )
    if sig_rows:
        concise += ["", "## Significance (same-class vs diff-class, both diff-seed)"]
        key_metrics = {("mask_iou_perm_maskmatch", "global"), ("cka_linear", "fc2"),
                       ("procrustes_distance", "fc2"), ("composition_accuracy", "global")}
        for sr in sig_rows:
            if (sr["metric"], sr["layer"]) in key_metrics:
                concise.append(
                    f"- {sr['metric']} ({sr['layer']}): Δ={sr['delta']:+.3f}, "
                    f"module-label perm-test p={sr['perm_p']:.3f} "
                    f"(Mann-Whitney p={sr['mannwhitney_p']:.3f}, descriptive)"
                )
        concise.append("Full table: significance_tests.csv")
    concise += [
        "",
        "## Functional",
    ]
    rew = [m.functional.get("rewarded_accuracy") for m in modules
           if m.functional.get("rewarded_accuracy") is not None
           and not np.isnan(m.functional.get("rewarded_accuracy", np.nan))]
    ent = [m.functional.get("nonrewarded_entropy") for m in modules
           if m.functional.get("nonrewarded_entropy") is not None
           and not np.isnan(m.functional.get("nonrewarded_entropy", np.nan))]
    if rew:
        concise.append(f"- mean rewarded accuracy: {np.mean(rew):.3f} (target ~0.99)")
    if ent:
        concise.append(f"- mean non-rewarded entropy: {np.mean(ent):.3f} (max ln10={np.log(10):.3f})")
    if composition:
        accs = [v["composition_accuracy"] for v in composition.values()]
        concise.append(f"- composition accuracy (all-class weight sum): {np.mean(accs):.3f} "
                       f"over {len(accs)} seed(s)")
    pc = _mean_for(rows, "diff_class_diff_seed", "composition_accuracy", "global")
    if pc is not None:
        concise.append(f"- pairwise merge accuracy (cross-class, diff-seed): {pc:.3f}")
        concise += ["", "## Does similarity predict pairwise composability? (Spearman)"]
        for key, label in [
            ("mask_iou_perm_maskmatch__global", "aligned IoU"),
            ("cka_linear__fc2", "CKA fc2"),
            ("procrustes_distance__fc2", "Procrustes fc2"),
            ("output_kl__global", "output KL"),
        ]:
            line = _spearman_line(pair_records, key, label)
            if line:
                concise.append(line)
    concise += ["", "## Figures", *[f"- figures/{f}" for f in figures]]
    (out_dir / "report_concise.md").write_text("\n".join(concise) + "\n", encoding="utf-8")

    # ---- full report ----
    full = [
        "# Calibrated MaxEnt module metrics - full report",
        "",
        f"- Generated: {utc_timestamp()} (elapsed {elapsed:.1f}s)",
        f"- Data source: {data_source}",
        f"- Classes: {classes}", f"- Seeds: {seeds}",
        f"- Architecture: DeepMLP 784->512->256->10",
        f"- MaxEnt lambda: {args.lambda_maxent}; IMP rounds: {args.prune_rounds}; "
        f"rate: {args.prune_rate}; epochs/round: {args.epochs_per_round}; lr: {args.lr}",
        f"- Eval examples: {args.eval_examples}; train subset: {args.train_subset}",
        f"- Modules: {n_modules}; total pairs: {n_pairs}",
        "",
        "## Methodological notes",
        "- Activations are the Linear-layer outputs (pre-ReLU for fc1/fc2, logits for fc3).",
        "- Permutation-aligned IoU uses Git Re-Basin style matching: `maskmatch` aligns",
        "  hidden units by weight/mask coordinate descent; `actmatch` aligns by activation",
        "  correlation on the shared eval inputs. fc1 is permuted by P1 (512 units), fc2 by",
        "  (P2 rows, P1 cols), fc3 by P2 (256 units).",
        "- Global IoU aggregates intersections/unions over all layers before the ratio,",
        "  so it is weighted by parameter count (fc3 does not dominate).",
        "- CKA/Procrustes follow Kornblith et al. (2019); interpret with Davari et al. (2023).",
        "- mask_spectral_dist compares singular-value spectra of the 0/1 layer masks:",
        "  permutation-invariant WITHOUT a matching step (no inflation bias); cospectral",
        "  caveat applies. mask_iou_null is the chance IoU of independent masks at the",
        "  pair's densities. Aligned metrics must be read against the random_baseline",
        "  (matching inflates chance similarity).",
        "- Pairwise composition: theta_merged = theta_a + theta_b incl. biases; accuracy is",
        "  unrestricted argmax on the pair's rewarded classes; interference = solo - merged.",
        "",
        "## Per-module functional summary",
        "",
        "| class | seed | rewarded acc | non-rew entropy | global sparsity |",
        "|---|---|---|---|---|",
    ]
    for m in modules:
        full.append(
            f"| {m.cls} | {m.seed} | {_fmt(m.functional.get('rewarded_accuracy'))} | "
            f"{_fmt(m.functional.get('nonrewarded_entropy'))} | "
            f"{_fmt(m.sparsity.get('global'))} |"
        )
    full += ["", "## Aggregated metrics by pair category / layer", "",
             "| pair_category | metric | layer | n | mean | std | min | max |",
             "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        full.append(
            f"| {r['pair_category']} | {r['metric']} | {r['layer']} | {r['n_pairs']} | "
            f"{r['mean']:.4f} | {r['std']:.4f} | {r['min']:.4f} | {r['max']:.4f} |"
        )
    if sig_rows:
        full += ["", "## Significance tests (same-class vs diff-class, diff-seed pairs)",
                 "",
                 "Pair-level Mann-Whitney p-values are descriptive (pairs share modules);",
                 "perm_p comes from the module-label permutation test, which respects that",
                 "dependence.",
                 "",
                 "| metric | layer | n_same | n_diff | mean_same | mean_diff | Δ | MW p | rank-biserial | perm p |",
                 "|---|---|---|---|---|---|---|---|---|---|"]
        for sr in sig_rows:
            full.append(
                f"| {sr['metric']} | {sr['layer']} | {sr['n_same']} | {sr['n_diff']} | "
                f"{sr['mean_same']:.4f} | {sr['mean_diff']:.4f} | {sr['delta']:+.4f} | "
                f"{_fmt(sr['mannwhitney_p'])} | {_fmt(sr['rank_biserial'])} | "
                f"{_fmt(sr['perm_p'])} |"
            )
    if composition:
        full += ["", "## Composition (all-class weight summation, theta_merged = sum theta_i)",
                 "",
                 "| seed | composition acc | composition entropy | n_eval |",
                 "|---|---|---|---|"]
        for seed, v in composition.items():
            full.append(f"| {seed} | {v['composition_accuracy']:.4f} | "
                        f"{v['composition_entropy']:.4f} | {v['n_eval']} |")
    barrier_recs = [r for r in pair_records
                    if r.get("barrier") and r["category"] == "same_class_diff_seed"]
    if barrier_recs:
        full += ["", "## Interpolation barrier (same-class diff-seed; raw vs weight-matched)",
                 "",
                 "| class | seeds | barrier loss raw | barrier loss aligned | acc drop raw | acc drop aligned |",
                 "|---|---|---|---|---|---|"]
        for r in barrier_recs:
            b = r["barrier"]
            full.append(
                f"| {r['class_a']} | {r['seed_a']}/{r['seed_b']} | "
                f"{b['barrier_loss_raw']:.4f} | {b['barrier_loss_aligned']:.4f} | "
                f"{b['barrier_accdrop_raw']:.4f} | {b['barrier_accdrop_aligned']:.4f} |"
            )
    full += ["", "## Figures", *[f"- figures/{f}" for f in figures]]
    (out_dir / "report_full.md").write_text("\n".join(full) + "\n", encoding="utf-8")


def _fmt(v) -> str:
    if v is None:
        return "n/a"
    try:
        if np.isnan(v):
            return "n/a"
    except (TypeError, ValueError):
        return str(v)
    return f"{v:.4f}"


def verify_controls(rows) -> bool:
    """Programmatic check of the calibration controls; prints PASS/FAIL lines.

    identical must be perfect; permuted_control must be recovered by the aligned
    and invariant metrics; random_baseline must stay clearly below the permuted
    ceiling. Returns True if all checks pass."""
    checks = [
        ("identical mask_iou_raw == 1",
         _mean_for(rows, "identical", "mask_iou_raw", "global"), lambda v: v > 0.999),
        ("identical output_mse == 0",
         _mean_for(rows, "identical", "output_mse", "global"), lambda v: v < 1e-9),
        ("permuted aligned IoU (maskmatch) ~= 1",
         _mean_for(rows, "permuted_control", "mask_iou_perm_maskmatch", "global"),
         lambda v: v > 0.99),
        ("permuted CKA fc2 ~= 1",
         _mean_for(rows, "permuted_control", "cka_linear", "fc2"), lambda v: v > 0.99),
        ("permuted Procrustes fc2 ~= 0",
         _mean_for(rows, "permuted_control", "procrustes_distance", "fc2"),
         lambda v: v < 0.01),
        ("permuted output_mse ~= 0",
         _mean_for(rows, "permuted_control", "output_mse", "global"), lambda v: v < 1e-6),
        ("permuted spectral dist ~= 0",
         _mean_for(rows, "permuted_control", "mask_spectral_dist", "global"),
         lambda v: v < 1e-9),
        ("random CKA fc2 below permuted ceiling",
         _mean_for(rows, "random_baseline", "cka_linear", "fc2"), lambda v: v < 0.9),
    ]
    all_ok = True
    for label, value, ok_fn in checks:
        if value is None:
            print(f"[controls] SKIP  {label}: no data")
            continue
        ok = ok_fn(value)
        all_ok &= ok
        print(f"[controls] {'PASS' if ok else 'FAIL'}  {label}: {value:.6f}")
    return all_ok

### Orchestration

Runs the whole workflow: train/load modules, compute pairs, aggregate, test, plot, and write reports.

In [ ]:
def make_train_loader_factory(train_x, train_y, args, device):
    def factory(loader_seed: int) -> DataLoader:
        g = torch.Generator()
        g.manual_seed(loader_seed)
        ds = TensorDataset(train_x, train_y)
        if args.train_subset and args.train_subset < train_x.size(0):
            idx = torch.randperm(train_x.size(0), generator=g)[: args.train_subset]
            ds = Subset(ds, idx.tolist())
        return DataLoader(ds, batch_size=args.batch_size, shuffle=True, generator=g)

    return factory


def run(args) -> None:
    start = time.time()
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = args.device
    classes = args.classes
    seeds = args.seeds

    print(f"[setup] out_dir={out_dir}")
    print(f"[setup] device={device} classes={classes} seeds={seeds}")

    train_ds, test_ds, data_source = get_datasets(args)
    train_x, train_y = stack_dataset(train_ds)
    test_x, test_y = stack_dataset(test_ds)
    eval_x, eval_y, eval_idx = build_eval_subset(test_x, test_y, args.eval_examples, seed=0)
    print(f"[data] source={data_source} train={tuple(train_x.shape)} "
          f"eval={tuple(eval_x.shape)} ({len(np.unique(eval_y))} classes present)")

    (out_dir / "activations").mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "activations" / "eval_labels.npy", eval_y.numpy())
    np.save(out_dir / "activations" / "eval_indices.npy", eval_idx)

    factory = make_train_loader_factory(train_x, train_y, args, device)

    # ---- Phase A: produce modules ----
    modules: list[Module] = []
    for cls in classes:
        for seed in seeds:
            tag = f"class{cls}_seed{seed}"
            if args.no_train or (module_is_complete(out_dir, cls, seed) and not args.overwrite):
                if module_is_complete(out_dir, cls, seed):
                    print(f"[module] {tag}: loading existing artifacts")
                    modules.append(load_module(out_dir, cls, seed, device))
                    continue
                if args.no_train:
                    print(f"[module] {tag}: SKIP (no checkpoint and --no-train)")
                    continue
            print(f"[module] {tag}: training ...")
            model, masks, info = imp_train_module(cls, seed, factory, args, device)
            functional = evaluate_module(model, eval_x, eval_y, cls, device)
            sparsity = layer_sparsity({k: v for k, v in masks.items()})
            acts = model.collect_activations(eval_x.to(device))
            save_module(out_dir, model, masks, cls, seed, functional, sparsity,
                        acts, info, args, data_source)
            modules.append(module_from_runtime(model, masks, cls, seed, functional, sparsity, acts))
            print(f"[module] {tag}: rew_acc={functional['rewarded_accuracy']:.3f} "
                  f"nonrew_ent={functional['nonrewarded_entropy']:.3f} "
                  f"sparsity={sparsity['global']:.3f}")

    if len(modules) < 1:
        print("ERROR: no modules available. Train some first (remove --no-train).",
              file=sys.stderr)
        return

    # ---- Phase B: pairs & metrics ----
    experimental_cats = ("same_class_diff_seed", "diff_class_same_seed", "diff_class_diff_seed")
    eval_labels = eval_y.numpy()
    pairs = build_pairs(modules, eval_x, args, device)
    print(f"[pairs] computing metrics for {len(pairs)} pairs ...")
    pair_records: list[dict[str, Any]] = []
    for i, ps in enumerate(pairs, 1):
        metrics = compute_pair_metrics(ps.a, ps.b, args.match_iters, eval_labels=eval_labels)
        record: dict[str, Any] = {
            "category": ps.category,
            "class_a": ps.class_a, "class_b": ps.class_b,
            "seed_a": ps.seed_a, "seed_b": ps.seed_b,
            "metrics": metrics,
        }
        if args.pair_compose and ps.category in experimental_cats:
            metrics.update(pairwise_composition_metrics(ps.a, ps.b, eval_x, eval_y, device))
        if args.barrier and ps.category in ("same_class_diff_seed", "permuted_control"):
            barrier = interpolation_barrier(
                ps.a, ps.b, eval_x, eval_y, args.lambda_maxent, device, args.match_iters)
            record["barrier"] = barrier
            for tag in ("raw", "aligned"):
                metrics[f"barrier_loss_{tag}__global"] = barrier[f"barrier_loss_{tag}"]
                metrics[f"barrier_accdrop_{tag}__global"] = barrier[f"barrier_accdrop_{tag}"]
        pair_records.append(record)
        if i % max(1, len(pairs) // 10) == 0 or i == len(pairs):
            print(f"[pairs]   {i}/{len(pairs)}")

    rows = aggregate(pair_records)
    controls_ok = verify_controls(rows)
    if not controls_ok:
        print("WARNING: at least one calibration control failed -- inspect before "
              "interpreting the experimental pair categories.", file=sys.stderr)
    sig_rows = significance_tests(pair_records, n_perm=args.n_perm)

    # ---- Phase C: composition (optional) ----
    composition = {}
    if args.compose and not any(m.cls < 0 for m in modules):
        composition = evaluate_composition(modules, classes, eval_x, eval_y, device)
        if composition:
            print(f"[compose] seeds with full class set: {list(composition.keys())}")

    # ---- Phase D: save results ----
    results_payload = {
        "config": {
            "classes": classes, "seeds": seeds, "data_source": data_source,
            "lambda_maxent": args.lambda_maxent, "prune_rounds": args.prune_rounds,
            "prune_rate": args.prune_rate, "epochs_per_round": args.epochs_per_round,
            "eval_examples": int(eval_x.size(0)), "match_iters": args.match_iters,
            "timestamp": utc_timestamp(),
        },
        "pairs": pair_records,
        "composition": composition,
    }
    (out_dir / "pairwise_results.json").write_text(
        json.dumps(results_payload, indent=2), encoding="utf-8")

    with (out_dir / "pairwise_summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["pair_category", "metric", "layer", "n_pairs", "mean", "std", "min", "max"])
        writer.writeheader()
        writer.writerows(rows)

    if sig_rows:
        with (out_dir / "significance_tests.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(sig_rows[0].keys()))
            writer.writeheader()
            writer.writerows(sig_rows)

    figures = make_all_plots(out_dir, rows, pair_records, classes)
    write_reports(out_dir, rows, pair_records, modules, classes, seeds, args,
                  data_source, composition, figures, time.time() - start, sig_rows)

    print(f"[done] wrote results to {out_dir}")
    print(f"[done]   pairwise_results.json, pairwise_summary.csv, "
          f"report_concise.md, report_full.md")
    print(f"[done]   figures: {figures}")
    print(f"[done] elapsed {time.time() - start:.1f}s")

### CLI

Defines default arguments reused by the Colab configuration cell without parsing notebook arguments.

In [ ]:
def int_list(text: str) -> list[int]:
    return [int(x) for x in text.replace(" ", "").split(",") if x != ""]


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--classes", type=int_list, default=[0, 1, 2, 3, 4],
                   help="rewarded classes, comma separated (default 0,1,2,3,4)")
    p.add_argument("--seeds", type=int_list, default=[42, 123, 456],
                   help="seeds, comma separated (default 42,123,456)")
    p.add_argument("--out-dir", default="raw/experiments/calibrated_module_metrics")
    p.add_argument("--data-root", type=Path, default=Path("data"))
    p.add_argument("--eval-examples", type=int, default=1280)
    # training / pruning
    p.add_argument("--lambda-maxent", type=float, default=1.0)
    p.add_argument("--prune-rounds", type=int, default=15)
    p.add_argument("--prune-rate", type=float, default=0.2)
    p.add_argument("--epochs-per-round", type=int, default=10)
    p.add_argument("--lr", type=float, default=1.2e-3)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--train-subset", type=int, default=None,
                   help="cap number of training examples (default: full set)")
    p.add_argument("--match-iters", type=int, default=6,
                   help="coordinate-descent iterations for weight matching")
    p.add_argument("--device", default="cpu")
    # modes
    p.add_argument("--overwrite", action="store_true", help="retrain even if artifacts exist")
    p.add_argument("--no-train", action="store_true",
                   help="only compute metrics from existing checkpoints/masks")
    p.add_argument("--compose", action="store_true",
                   help="evaluate all-class weight-sum composition per seed (needs all classes)")
    p.add_argument("--no-pair-compose", dest="pair_compose", action="store_false",
                   help="skip pairwise (theta_a + theta_b) composition metrics")
    p.add_argument("--barrier", action="store_true",
                   help="compute interpolation barriers (raw vs weight-matched) for "
                        "same-class diff-seed pairs and permuted controls")
    p.add_argument("--n-perm", type=int, default=1000,
                   help="module-label permutations for the significance test")
    p.add_argument("--smoke-test", action="store_true",
                   help="tiny end-to-end run (overrides grid/training sizes)")
    return p


def apply_smoke_test(args) -> None:
    args.classes = [0, 1]
    args.seeds = [42, 123]
    args.prune_rounds = 2
    args.epochs_per_round = 1
    args.eval_examples = 256
    args.train_subset = 2000
    args.batch_size = 128
    args.match_iters = 4
    args.compose = True
    args.barrier = True
    args.n_perm = 200
    if "--out-dir" not in sys.argv:
        args.out_dir = "raw/experiments/calibrated_module_metrics_smoke"

## Configuration

- `SMOKE = True` -> tiny end-to-end check (2 classes x 2 seeds, 2 IMP rounds).
- `SMOKE = False`, `FULL_10X10 = False` -> minimal grid
  (classes 0-4 x seeds 42/123/456).
- `SMOKE = False`, `FULL_10X10 = True` -> full grid
  (10 classes x 10 seeds = 100 modules).

Useful overrides (uncomment in the cell):
- `args.epochs_per_round = 5` — faster exploratory variant; keep `10` for the
  main reported run unless you deliberately decide otherwise.
- `args.no_train = True` — recompute all metrics from existing checkpoints (no training).
- `args.overwrite = True` — retrain even if a module's artifacts exist.
- `args.barrier = True` — optional interpolation barriers; expensive on 100 modules.


In [ ]:
# This cell is the only place you normally edit before running the experiment.
# First run: keep SMOKE=True and verify every calibration control prints PASS.
# Full run: set SMOKE=False and FULL_10X10=True.
SMOKE = True
FULL_10X10 = False

# Keep all defaults in one place: this mirrors the canonical command-line script.
args = build_parser().parse_args([])
args.device = "cuda" if torch.cuda.is_available() else "cpu"
args.data_root = Path("/content/data") if IN_COLAB else (Path("/kaggle/working/data") if IN_KAGGLE else Path("data"))
args.compose = True      # all-class weight-sum composition per seed
args.barrier = False     # real-run default: optional and expensive on 100 modules

# Optional overrides. Uncomment intentionally, then leave a note in your report.
# args.epochs_per_round = 5
# args.no_train = True
# args.overwrite = True
# args.barrier = True

if SMOKE:
    apply_smoke_test(args)
    args.out_dir = str(BASE_DIR / "calibrated_module_metrics_smoke")
else:
    if FULL_10X10:
        args.classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
        args.seeds = [42, 123, 456, 789, 1024, 2024, 31415, 27182, 16180, 9999]
        args.out_dir = str(BASE_DIR / "calibrated_module_metrics_full_10x10")
    else:
        args.out_dir = str(BASE_DIR / "calibrated_module_metrics_minimal_5x3")

print(f"device={args.device}")
print(f"out_dir={args.out_dir}")
print(f"classes={args.classes}  seeds={args.seeds}")
print(f"IMP: {args.prune_rounds} rounds x {args.prune_rate:.0%}, "
      f"epochs/round={args.epochs_per_round}, lambda={args.lambda_maxent}")
print(f"modules to produce: {len(args.classes) * len(args.seeds)}")
print(f"pairwise composition={args.pair_compose}, all-class composition={args.compose}, "
      f"barrier={args.barrier}, n_perm={args.n_perm}")


## Run

Resume-safe: already-completed modules are skipped.

In [ ]:
# This cell launches the full pipeline with the configuration printed above.
run(args)


## Concise report

Displays the short summary written by the pipeline.

In [ ]:
# This cell displays the short report generated by the run.
from IPython.display import Markdown, display

display(Markdown((Path(args.out_dir) / "report_concise.md").read_text(encoding="utf-8")))


## Figures

Displays every generated figure so the main patterns can be inspected inside the notebook.

In [ ]:
# This cell displays every PNG figure saved by the pipeline.
from IPython.display import Image, Markdown, display

fig_dir = Path(args.out_dir) / "figures"
for p in sorted(fig_dir.glob("*.png")):
    display(Markdown(f"### {p.name}"))
    display(Image(filename=str(p)))


## After the run

1. **Check the controls first** (top of the report above): every line must be
   the expected pattern — `identical` perfect, `permuted_control` recovered by
   the aligned/invariant metrics, `random_baseline` low. The training log also
   prints `[controls] PASS/FAIL` lines.
2. Read the three signal sections (structural / representational / functional)
   and `significance_tests.csv` (the module-label `perm_p` column is the
   honest one — pair-level p-values overstate evidence).
3. Share the whole output folder plus these entry points:
   - `report_concise.md` — short narrative summary;
   - `report_full.md` — complete tables and methodological notes;
   - `pairwise_summary.csv` — means, standard deviations, min/max by category;
   - `significance_tests.csv` — same-class vs cross-class tests;
   - `pairwise_results.json` — full pair-level data;
   - `figures/` — all generated plots.
4. Bring the results back into the thesis repo: download the output folder
   from Drive/Kaggle outputs into `MSc_thesis/raw/experiments/`.
   Metrics can be recomputed locally from those checkpoints with
   `python3 scripts/run_calibrated_module_experiments.py --no-train ...`.
